# Paper 3 - 00 . Pilot smoke test (gates bulk Stage 2)

Per PAPER3_PLAN section 15, this notebook is the gate that protects bulk
OpenRouter spend in notebook 02. It runs a 50-prompt teacher + judge
round-trip (about $0.30 observed) and writes
`data/preferences/<short>/smoke.json` for all three anchors only if
every gate passes. Notebook 02's pre-flight cell refuses to start until
the smoke file exists with `smoke_ok=True` and a timestamp within the
last 7 days, and the locked-prompt SHA-256 digests recorded by this
notebook match the live ones in `src/prompts.py`.

Five gates checked (the v3 design after dropping a misframed judge-
agreement gate; full rationale in EXPERIMENT_LOG.md):

1. **Locked prompts.** Prompts are imported from `src/prompts.py` and
   digests are printed for the smoke record. Notebook 02's pre-flight
   cell verifies these against the live module.
2. **Teacher cooperation with research framing.** Refusal-on-meta-task
   rate at most 10% on harmful side (Paper 2 R8 lesson).
3. **Reasoning-token overrun.** No teacher response with reasoning
   tokens > 50% of `max_tokens` (Paper 2 R10 lesson).
4. **Cost overrun (one-sided).** Observed cost is at most 30% above
   the ledger estimate. Cheaper than expected is fine.
5. **Bulk projection.** Projection at the *observed* cost rate is at
   most the OpenRouter cap ($200).

Per-pair audit JSONL is written for ad-hoc inspection. Per-side and
per-dim agreement are still computed and printed but flagged
DIAGNOSTIC-ONLY -- they do not affect the gate decision.

**Smoke cost.** ~50 mixed teacher calls + ~50 judge calls. Observed
headline ~$0.20-0.30 at May 2026 OpenRouter prices.

In [ ]:
%%capture
!pip install -U \
    'requests>=2.32' \
    'tenacity>=9.0' \
    'pyyaml>=6.0' \
    huggingface_hub ipywidgets -q

In [ ]:
import os, json, gc, time, hashlib, sys, subprocess
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
REPO_ROOT   = Path("/content/rosafety-align")

PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SMOKE_DIR    = DRIVE_ROOT / "data" / "smoke"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [PREFS_DIR, SMOKE_DIR, SPLITS_DIR, ADAPTERS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Pull src/ from GitHub so src.prompts is the single source of truth ---
# Notebook 00 writes smoke.json with the digests it computed at run time;
# notebook 02 verifies those digests against the live module here. Without
# a centralised source the two cells could drift, silently invalidating the
# OpenRouter cache (PAPER3_PLAN section 15.2).
if not REPO_ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/robery567/rosafety-align.git",
        str(REPO_ROOT),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=False)

# --- Paper 2 reuse: judges, llm_judge, datasheet helpers ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- Paper 3 single-source-of-truth helpers ---
sys.path.insert(0, str(REPO_ROOT / "src"))
from prompts import (
    TEACHER_SYSTEM_HARMFUL, TEACHER_SYSTEM_BENIGN,
    JUDGE_SYSTEM, JUDGE_USER_TEMPLATE,
    PROMPT_DIGESTS, CACHE_NAMESPACE_VERSION,
    sha16, short_of, family_of,
    verify_smoke_gate, display_locked_state, SmokeGateNotPassed,
    SMOKE_FRESHNESS_DAYS,
)

# --- OpenRouter key from Colab Secrets ---
try:
    from google.colab import userdata
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
except Exception as _e:
    print(f"(secrets not configured: {_e})")

assert os.environ.get("OPENROUTER_API_KEY"), \
    "Set OPENROUTER_API_KEY in Colab Secrets before running this notebook."

print("Bootstrap done.")
print(display_locked_state())

## Configuration

In [ ]:
ANCHORS_TO_GATE = [
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "google/gemma-3-4b-it",
]

SMOKE_HARMFUL_TARGET = {"toxicity": 16, "jailbreak": 12, "crosslingual": 12}
SMOKE_BENIGN_TARGET  = {"overrefusal": 10}

TEACHER_HARMFUL = "anthropic/claude-opus-4.7"
TEACHER_BENIGN  = "meta-llama/Llama-3.3-70B-Instruct"
JUDGE_MODEL     = "openai/gpt-5-mini"

TEACHER_MAX_TOKENS = 1024
JUDGE_MAX_TOKENS   = 600
TEACHER_WORKERS    = 4
JUDGE_WORKERS      = 8

PRICE_PER_CALL = {
    TEACHER_HARMFUL: 0.025,
    TEACHER_BENIGN:  0.0005,
    JUDGE_MODEL:     0.001,
}

GATE_TEACHER_REFUSAL_MAX = 0.10
GATE_REASONING_OVERRUN   = 0.50
GATE_COST_OVERRUN_MAX    = 0.30
OPENROUTER_CAP_USD       = 200.0

BULK_HARMFUL_PER_ANCHOR = 600
BULK_BENIGN_PER_ANCHOR  = 200
BULK_JUDGE_PER_ANCHOR   = 3200

SMOKE_SET_PATH = SMOKE_DIR / "smoke_set_v2.jsonl"
SMOKE_VERSION  = "v3"
print(f"smoke set : {SMOKE_SET_PATH}")
print(f"version   : {SMOKE_VERSION}")
print(f"anchors to gate: {len(ANCHORS_TO_GATE)}")

## Sample 50 stratified prompts

First run constructs and pins the smoke set; later runs read it back.

In [ ]:
if SMOKE_SET_PATH.exists():
    smoke_set = [json.loads(l) for l in open(SMOKE_SET_PATH, encoding="utf-8")]
    print(f"loaded frozen smoke set ({len(smoke_set)} prompts) from {SMOKE_SET_PATH}")
else:
    import random
    rng = random.Random(0xCAFEBABE)
    bench_dir = PAPER2_ROOT / "benchmark" / "expanded"
    smoke_set = []
    for dim, n in {**SMOKE_HARMFUL_TARGET, **SMOKE_BENIGN_TARGET}.items():
        path = bench_dir / f"{dim}.jsonl"
        prompts_dim = [json.loads(l) for l in open(path, encoding="utf-8")]
        rng.shuffle(prompts_dim)
        for x in prompts_dim[:n]:
            smoke_set.append({
                "id": x["id"], "dim": dim,
                "side": "benign" if dim in SMOKE_BENIGN_TARGET else "harmful",
                "text_ro": x["text_ro"], "category": x.get("category", ""),
            })
    rng.shuffle(smoke_set)
    with open(SMOKE_SET_PATH, "w", encoding="utf-8") as f:
        for x in smoke_set: f.write(json.dumps(x, ensure_ascii=False) + "\n")
    print(f"constructed smoke set ({len(smoke_set)} prompts) -> {SMOKE_SET_PATH}")

SMOKE_SET_SHA = hashlib.sha256(SMOKE_SET_PATH.read_bytes()).hexdigest()[:16]
print(f"  sha256[:16] = {SMOKE_SET_SHA}")

from collections import Counter
print(f"  composition : {dict(Counter(x['dim'] for x in smoke_set))}")

## OpenRouter wrapper

In [ ]:
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
SESSION = requests.Session()
SESSION.headers.update({
    "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
    "Content-Type":  "application/json",
    "HTTP-Referer":  "https://github.com/anonymised/rosafety-align",
    "X-Title":       "RoSafetyAlign",
})

@retry(stop=stop_after_attempt(4),
       wait=wait_exponential(multiplier=1, min=2, max=20),
       retry=retry_if_exception_type((requests.exceptions.RequestException, RuntimeError)))
def or_chat_completion(*, model, system, user, max_tokens, temperature=0.0,
                        response_format=None):
    payload = {
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "usage": {"include": True},
    }
    if response_format is not None:
        payload["response_format"] = response_format
    if model.startswith("openai/gpt-5"):
        payload["reasoning"] = {"effort": "minimal"}

    r = SESSION.post(OPENROUTER_URL, json=payload, timeout=120)
    if r.status_code == 429:
        raise RuntimeError("429 rate limited")
    r.raise_for_status()
    data = r.json()

    choice = data["choices"][0]
    text   = choice["message"].get("content", "") or ""
    finish = choice.get("finish_reason", "")
    native = choice.get("native_finish_reason", "")

    usage   = data.get("usage", {}) or {}
    prompt_tokens     = int(usage.get("prompt_tokens",     0) or 0)
    completion_tokens = int(usage.get("completion_tokens", 0) or 0)
    details = usage.get("completion_tokens_details") or {}
    reasoning_tokens  = int(details.get("reasoning_tokens", 0) or 0)
    cost_usd = float(usage.get("cost", 0.0) or 0.0)

    return {
        "text": text,
        "finish_reason": finish,
        "native_finish_reason": native,
        "usage": {
            "prompt_tokens":     prompt_tokens,
            "completion_tokens": completion_tokens,
            "reasoning_tokens":  reasoning_tokens,
            "cost_usd":          cost_usd,
        },
    }

def or_balance() -> float:
    r = SESSION.get("https://openrouter.ai/api/v1/credits", timeout=30)
    if r.status_code != 200:
        return float("nan")
    data = r.json().get("data", {}) or {}
    total = float(data.get("total_credits", 0.0))
    used  = float(data.get("total_usage",   0.0))
    return total - used

def parse_judge_json(text: str) -> dict:
    """Parse the binary refusal/compliance JSON, with a tolerant fallback
    that pulls the first balanced {...} out of the response."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    if start == -1:
        return {"label": "compliance", "justification": "judge JSON parse failed"}
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try: return json.loads(text[start:i+1])
                except json.JSONDecodeError: continue
    return {"label": "compliance", "justification": "judge JSON parse failed"}

bal_before = or_balance()
print(f'OpenRouter balance before smoke: ${bal_before:.4f}')

## Teacher pipeline

In [ ]:
def call_teacher(item):
    if item["side"] == "harmful":
        model, system = TEACHER_HARMFUL, TEACHER_SYSTEM_HARMFUL
    else:
        model, system = TEACHER_BENIGN, TEACHER_SYSTEM_BENIGN
    try:
        t0 = time.time()
        resp = or_chat_completion(
            model=model, system=system, user=item["text_ro"],
            max_tokens=TEACHER_MAX_TOKENS, temperature=0.0,
        )
        resp["wallclock_s"] = round(time.time() - t0, 2)
        resp["model"]       = model
        return item["id"], resp
    except Exception as e:
        return item["id"], {"error": str(e), "model": model}

print(f"running teacher pipeline ({len(smoke_set)} calls, workers={TEACHER_WORKERS}) ...")
teacher_results = {}
t0 = time.time()
with ThreadPoolExecutor(max_workers=TEACHER_WORKERS) as pool:
    futures = [pool.submit(call_teacher, x) for x in smoke_set]
    for i, fut in enumerate(as_completed(futures), 1):
        item_id, resp = fut.result()
        teacher_results[item_id] = resp
        if i % 10 == 0:
            print(f"  {i:>3}/{len(smoke_set)}  ({time.time() - t0:.1f}s)")
print(f"teacher done in {time.time() - t0:.1f}s")
errors = [(i, r) for i, r in teacher_results.items() if "error" in r]
if errors:
    print(f"\nTEACHER ERRORS: {len(errors)}")
    for item_id, r in errors[:5]:
        print(f"  {item_id}  {r['model']}  {r['error'][:120]}")

## Judge pipeline (verification)

Diagnostic only -- per the v3 redesign, judge agreement does not gate
the smoke run. The judgements still populate the per-pair audit and the
diagnostic dashboard.

In [ ]:
def call_judge(item):
    teacher = teacher_results.get(item["id"], {})
    text = teacher.get("text") or "[empty response]"
    try:
        resp = or_chat_completion(
            model=JUDGE_MODEL,
            system=JUDGE_SYSTEM,
            user=JUDGE_USER_TEMPLATE.format(prompt=item["text_ro"], response=text),
            max_tokens=JUDGE_MAX_TOKENS, temperature=0.0,
            response_format={"type": "json_object"},
        )
        parsed = parse_judge_json(resp["text"])
        resp["label"] = parsed.get("label", "compliance")
        resp["justification"] = parsed.get("justification", "")
        return item["id"], resp
    except Exception as e:
        return item["id"], {"error": str(e), "model": JUDGE_MODEL}

print(f"running judge pipeline ({len(smoke_set)} calls, workers={JUDGE_WORKERS}) ...")
judge_results = {}
t0 = time.time()
with ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as pool:
    futures = [pool.submit(call_judge, x) for x in smoke_set]
    for i, fut in enumerate(as_completed(futures), 1):
        item_id, resp = fut.result()
        judge_results[item_id] = resp
        if i % 10 == 0:
            print(f"  {i:>3}/{len(smoke_set)}  ({time.time() - t0:.1f}s)")
print(f"judge done in {time.time() - t0:.1f}s")

## Per-pair audit log

In [ ]:
audit_records = []
for x in smoke_set:
    t = teacher_results.get(x["id"], {})
    j = judge_results.get(x["id"], {})
    expected = "refusal" if x["side"] == "harmful" else "compliance"
    usage = t.get("usage", {}) or {}
    audit_records.append({
        "id": x["id"], "dim": x["dim"], "side": x["side"],
        "category": x.get("category", ""),
        "expected": expected,
        "judge_label": j.get("label", "?"),
        "agree": j.get("label") == expected,
        "prompt_preview": x["text_ro"][:300],
        "teacher_preview": (t.get("text") or t.get("error") or "")[:600],
        "teacher_finish_reason": t.get("finish_reason", ""),
        "teacher_tokens": usage.get("completion_tokens", 0),
        "reasoning_tokens": usage.get("reasoning_tokens", 0),
        "judge_justification": j.get("justification", ""),
    })

audit_path = SMOKE_DIR / f"smoke_audit_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
with open(audit_path, "w", encoding="utf-8") as f:
    for r in audit_records: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"per-pair audit written: {audit_path}")

## Pair display

First 14 pairs side-by-side. Disagreements are highlighted in amber;
agreements in green. Disagreements are diagnostic only.

In [ ]:
from IPython.display import HTML, display
rows = audit_records[:14]
html = "<style>td{padding:6px;vertical-align:top;font-family:system-ui,sans-serif;font-size:12px;border-bottom:1px solid #e5e7eb}th{padding:6px;background:#1f2937;color:#fff;text-align:left}.ok{background:#d1fae5}.bad{background:#fef3c7}</style>"
html += "<table style='border-collapse:collapse'><tr><th>side</th><th>dim</th><th>expected</th><th>actual</th><th>tokens</th><th>prompt</th><th>teacher response</th><th>judge says</th></tr>"
for r in rows:
    cls = "ok" if r["agree"] else "bad"
    html += (f"<tr class='{cls}'>"
             f"<td>{r['side']}</td><td>{r['dim']}</td>"
             f"<td>{r['expected']}</td><td>{r['judge_label']}</td>"
             f"<td>{r['teacher_tokens']}<br><small>({r['reasoning_tokens']}r)</small></td>"
             f"<td>{r['prompt_preview']}</td><td>{r['teacher_preview']}</td>"
             f"<td>{r['judge_justification']}</td></tr>")
html += "</table>"
display(HTML(html))

## Gate metrics + decision

In [ ]:
harmful_items = [x for x in smoke_set if x["side"] == "harmful"]
benign_items  = [x for x in smoke_set if x["side"] == "benign"]

teacher_refusal_count = 0
for x in harmful_items:
    text = (teacher_results.get(x["id"], {}).get("text") or "").strip()
    if len(text) < 20:
        teacher_refusal_count += 1
teacher_refusal_rate = teacher_refusal_count / max(len(harmful_items), 1)

reasoning_overruns = 0
for x in smoke_set:
    rtok = (teacher_results.get(x["id"], {}).get("usage") or {}).get("reasoning_tokens", 0)
    if rtok > GATE_REASONING_OVERRUN * TEACHER_MAX_TOKENS:
        reasoning_overruns += 1

def side_agreement(items):
    n = correct = 0
    for x in items:
        j = judge_results.get(x["id"], {})
        if "error" in j or "label" not in j: continue
        n += 1
        expected = "refusal" if x["side"] == "harmful" else "compliance"
        if j["label"] == expected: correct += 1
    return correct, n, (correct / n if n else 0.0)

h_correct, h_n, h_agree = side_agreement(harmful_items)
b_correct, b_n, b_agree = side_agreement(benign_items)
overall_correct = h_correct + b_correct
overall_n       = h_n + b_n
judge_agreement = overall_correct / max(overall_n, 1)

from collections import defaultdict
per_dim = defaultdict(lambda: [0, 0])
for r in audit_records:
    per_dim[r["dim"]][1] += 1
    if r["agree"]: per_dim[r["dim"]][0] += 1

bal_after = or_balance()
observed_cost = (bal_before - bal_after) if (bal_before == bal_before and bal_after == bal_after) else float("nan")
estimated_cost = (
    len(harmful_items) * PRICE_PER_CALL[TEACHER_HARMFUL] +
    len(benign_items)  * PRICE_PER_CALL[TEACHER_BENIGN]  +
    len(smoke_set)     * PRICE_PER_CALL[JUDGE_MODEL]
)
if observed_cost == observed_cost:
    cost_overrun = max(0.0, (observed_cost - estimated_cost) / max(estimated_cost, 1e-9))
    scale = (BULK_HARMFUL_PER_ANCHOR + BULK_BENIGN_PER_ANCHOR + BULK_JUDGE_PER_ANCHOR) / len(smoke_set)
    bulk_per_anchor    = observed_cost * scale
    bulk_three_anchors = bulk_per_anchor * 3
else:
    cost_overrun = 0.0
    bulk_three_anchors = (
        BULK_HARMFUL_PER_ANCHOR * PRICE_PER_CALL[TEACHER_HARMFUL] +
        BULK_BENIGN_PER_ANCHOR  * PRICE_PER_CALL[TEACHER_BENIGN]  +
        BULK_JUDGE_PER_ANCHOR   * PRICE_PER_CALL[JUDGE_MODEL]
    ) * 3

def emit(passed, label, value, threshold):
    glyph = "PASS" if passed else "FAIL"
    print(f"  [{glyph}]  {label:<46} {value:>14}   threshold {threshold}")

print("=" * 84)
print("GATE DASHBOARD")
print("=" * 84)

g1 = True
print(f"  [PASS]  Locked-prompt digests recorded                 (4 SHAs)")

g2 = teacher_refusal_rate <= GATE_TEACHER_REFUSAL_MAX
emit(g2, "Teacher refusal-on-research rate (harmful side)",
     f"{teacher_refusal_rate:.0%}", f"<= {GATE_TEACHER_REFUSAL_MAX:.0%}")

g3 = reasoning_overruns == 0
emit(g3, "Reasoning-token overruns (count)", f"{reasoning_overruns}", "= 0")

g4 = cost_overrun <= GATE_COST_OVERRUN_MAX
cost_str = (f"${observed_cost:.3f} (est ${estimated_cost:.3f})"
            if observed_cost == observed_cost else "no balance read")
emit(g4, "Cost overrun vs ledger (one-sided)", cost_str,
     f"overrun <= {GATE_COST_OVERRUN_MAX:.0%}")

g5 = bulk_three_anchors <= OPENROUTER_CAP_USD
emit(g5, "Bulk projection at observed rate (3 anchors)",
     f"${bulk_three_anchors:.2f}", f"<= ${OPENROUTER_CAP_USD:.0f}")

print("-" * 84)
print("DIAGNOSTIC-ONLY (does not gate)")
print(f"  judge agreement overall: {judge_agreement:.0%}  ({overall_correct}/{overall_n})")
print(f"           harmful side  : {h_agree:.0%}  ({h_correct}/{h_n})")
print(f"           benign  side  : {b_agree:.0%}  ({b_correct}/{b_n})")
print(f"  per-dim agreement:")
for dim, (c, n) in sorted(per_dim.items()):
    print(f"    {dim:<13s}  {c}/{n}  ({c/max(n,1):.0%})")

print("=" * 84)
smoke_ok = all([g1, g2, g3, g4, g5])
print(f"  smoke_ok = {smoke_ok}")
print("=" * 84)

## Decision

In [ ]:
smoke_record = {
    "smoke_ok": bool(smoke_ok),
    "smoke_set_path": str(SMOKE_SET_PATH),
    "smoke_set_sha256_16": SMOKE_SET_SHA,
    "smoke_set_version": SMOKE_VERSION,
    "audit_path": str(audit_path),
    "prompt_digests": dict(PROMPT_DIGESTS),
    "cache_namespace_version": CACHE_NAMESPACE_VERSION,
    "anchors_gated": ANCHORS_TO_GATE,
    "models": {
        "teacher_harmful": TEACHER_HARMFUL,
        "teacher_benign":  TEACHER_BENIGN,
        "judge":           JUDGE_MODEL,
    },
    "gates": {
        "teacher_refusal_rate":    round(teacher_refusal_rate, 4),
        "teacher_refusal_max":     GATE_TEACHER_REFUSAL_MAX,
        "reasoning_overruns":      reasoning_overruns,
        "observed_cost_usd":       round(observed_cost, 4) if observed_cost == observed_cost else None,
        "estimated_cost_usd":      round(estimated_cost, 4),
        "cost_overrun":            round(cost_overrun, 4),
        "cost_overrun_max":        GATE_COST_OVERRUN_MAX,
        "bulk_three_anchors_usd":  round(bulk_three_anchors, 2),
        "openrouter_cap_usd":      OPENROUTER_CAP_USD,
    },
    "diagnostic": {
        "judge_agreement_overall": round(judge_agreement, 4),
        "judge_agreement_harmful": round(h_agree, 4),
        "judge_agreement_benign":  round(b_agree, 4),
        "per_dim_agreement":       {d: {"agree": c, "n": n} for d, (c, n) in per_dim.items()},
    },
    "smoke_set_size": len(smoke_set),
    "completed_at":   datetime.now(timezone.utc).isoformat(timespec="seconds"),
}

if smoke_ok:
    for anchor_id in ANCHORS_TO_GATE:
        short = short_of(anchor_id)
        anchor_dir = PREFS_DIR / short
        anchor_dir.mkdir(parents=True, exist_ok=True)
        (anchor_dir / "smoke.json").write_text(json.dumps(smoke_record, indent=2))
        print(f"  wrote smoke.json -> {anchor_dir / 'smoke.json'}")
    print(f"\nSmoke gate PASSED. Notebook 02 is unblocked for the next "
          f"{SMOKE_FRESHNESS_DAYS} days.")
else:
    fail_path = SMOKE_DIR / f"smoke_failed_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
    fail_path.write_text(json.dumps(smoke_record, indent=2))
    print(f"\nSmoke gate FAILED. Diagnostic: {fail_path}")
    print(f"Per-pair audit:           {audit_path}")
    print("Address the failing gate(s) and re-run.")

print("\n" + json.dumps(smoke_record, indent=2))